# HW 2 — Applied: Digit Addition
**Modern Computer Vision** — Technion, Spring 2026

**Due:** May 10, 2026

**Student ID:**

In [ ]:
STUDENT_ID = ""  # <-- Fill in your student ID

---
## Overview

This assignment has two phases:

**Phase 1 — Train a digit classifier.**  
Train a neural network to classify individual MNIST digits (0–9) using PyTorch.
This is your standard supervised learning pipeline: data loading, model, training loop, evaluation.

**Phase 2 — Digit addition, without training.**  
Given a *pair* of digit images, predict their **sum** (a number in 0–18).
The catch: **you may not train anything new.** No fine-tuning, no new loss functions, no gradient updates.
You can only use the classifier from Phase 1 and any non-learned operations you come up with.

Think carefully about what your classifier gives you and how to combine two predictions.
There is an elegant mathematical solution — but creative approaches are welcome too.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline

# Use GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

---
## Phase 1: Train a digit classifier

Train a model that takes a 28×28 grayscale image and predicts which digit (0–9) it is.
Use whatever architecture you like — an MLP is fine, a small CNN is also fine.
Aim for **at least 97% test accuracy** before moving to Phase 2.

### Data

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_data = torchvision.datasets.MNIST('./data', train=True, download=True, transform=transform)
test_data  = torchvision.datasets.MNIST('./data', train=False, transform=transform)

train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_data,  batch_size=1000, shuffle=False)

# Visualize a few samples
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for i, ax in enumerate(axes):
    img, label = train_data[i]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(str(label))
    ax.axis('off')
plt.suptitle('MNIST samples')
plt.show()

### Your classifier

Define your model, optimizer, and training loop. You're free to choose the architecture.

In [ ]:
# YOUR CODE HERE — define your model
# Example skeleton:
#
# class DigitClassifier(nn.Module):
#     def __init__(self):
#         super().__init__()
#         ...
#     def forward(self, x):
#         ...
#         return logits   # (B, 10) — raw scores, not probabilities
#
# model = DigitClassifier().to(device)
# optimizer = ...

raise NotImplementedError

In [ ]:
# YOUR CODE HERE — training loop
# Train until you reach at least 97% test accuracy.
# Store loss and accuracy per epoch in lists for plotting below.
# Remember to move data to `device`: images, labels = images.to(device), labels.to(device)
#
# Example structure:
# train_losses = []
# test_accs = []
# for epoch in range(num_epochs):
#     model.train()
#     ...
#     model.eval()
#     ...
#     train_losses.append(avg_loss)
#     test_accs.append(accuracy)

raise NotImplementedError

In [ ]:
# Evaluate and plot training curves

# --- Test accuracy ---
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        preds = model(images).argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
test_accuracy = correct / total
print(f"Test accuracy: {test_accuracy:.2%}")
assert test_accuracy >= 0.97, f"Need at least 97% accuracy, got {test_accuracy:.2%}"

# --- Training curves ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(range(1, len(train_losses)+1), train_losses, 'o-')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Training Loss')
ax1.set_title('Loss')
ax1.grid(True, alpha=0.3)

ax2.plot(range(1, len(test_accs)+1), [a*100 for a in test_accs], 'o-', color='green')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Test Accuracy (%)')
ax2.set_title('Accuracy')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# --- Sample predictions ---
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i in range(8):
    img, label = test_data[i]
    with torch.no_grad():
        logits = model(img.unsqueeze(0).to(device))
    pred = logits.argmax(dim=1).item()
    probs = torch.softmax(logits, dim=1).squeeze().cpu()
    
    axes[0, i].imshow(img.squeeze(), cmap='gray')
    axes[0, i].set_title(f'{pred}', color='green' if pred == label else 'red', fontweight='bold')
    axes[0, i].axis('off')
    
    axes[1, i].bar(range(10), probs.numpy(), color=['green' if j == label else 'steelblue' for j in range(10)])
    axes[1, i].set_ylim(0, 1)
    axes[1, i].set_xticks(range(10))
    axes[1, i].tick_params(labelsize=7)

plt.suptitle('Sample predictions', fontsize=13)
plt.tight_layout()
plt.show()

---
## Phase 2: Digit addition — without training

Now for the interesting part.

You receive **pairs** of digit images. For each pair, you need to predict their **sum** — a number between 0 and 18.

**The constraint:** you may NOT train any new model, fine-tune anything, or compute any gradients.
You can only use:
- Your trained classifier from Phase 1 (in eval mode, no gradients)
- Any mathematical/programmatic operations that don't involve learning

Think about it: your classifier gives you something useful about each digit.
How can you combine that information to predict the sum?

*Hint: think about what we studied in HW1 theory about combining supports under convolution.*

### The addition dataset

We provide a dataset that pairs two random MNIST images and labels each pair with their sum.

In [ ]:
class DigitAdditionDataset(Dataset):
    """Pairs of MNIST digits with sum labels (0-18)."""
    
    def __init__(self, mnist_dataset, n_pairs=10000, seed=42):
        self.mnist = mnist_dataset
        rng = np.random.RandomState(seed)
        self.pairs = rng.randint(0, len(mnist_dataset), (n_pairs, 2))
    
    def __len__(self):
        return len(self.pairs)
    
    def __getitem__(self, idx):
        i, j = self.pairs[idx]
        img1, label1 = self.mnist[i]
        img2, label2 = self.mnist[j]
        return img1, img2, label1 + label2  # sum is in [0, 18]


addition_test = DigitAdditionDataset(test_data, n_pairs=10000, seed=0)
addition_loader = DataLoader(addition_test, batch_size=256, shuffle=False)

# Visualize some pairs
fig, axes = plt.subplots(2, 6, figsize=(12, 4))
for i in range(6):
    img1, img2, s = addition_test[i]
    axes[0, i].imshow(img1.squeeze(), cmap='gray')
    axes[0, i].axis('off')
    axes[1, i].imshow(img2.squeeze(), cmap='gray')
    axes[1, i].axis('off')
    axes[0, i].set_title(f'sum = {s}', fontsize=11)
plt.suptitle('Digit addition: predict the sum of each pair', fontsize=13)
plt.tight_layout()
plt.show()

### Your approach

Implement a function that takes two digit images and predicts their sum.

**Rules:**
- You may call your trained classifier (in `eval` mode, inside `torch.no_grad()`)
- You may NOT use `loss.backward()`, `optimizer.step()`, or train anything
- You may use any non-learned operations (math, numpy, torch ops, etc.)

Explain your approach in the markdown cell below.

**Your approach (explain here):**

*Describe your method and why it works.*

In [ ]:
def predict_sum(model, img1, img2):
    """
    Predict the sum of two digit images.
    
    Args:
        model: your trained digit classifier (in eval mode)
        img1:  (B, 1, 28, 28) batch of first digits
        img2:  (B, 1, 28, 28) batch of second digits
    Returns:
        predicted_sums: (B,) integer predictions in [0, 18]
    """
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# Evaluate your approach
model.eval()
correct = 0
total = 0
all_preds = []
all_targets = []

with torch.no_grad():
    for img1, img2, targets in addition_loader:
        img1, img2 = img1.to(device), img2.to(device)
        preds = predict_sum(model, img1, img2)
        if isinstance(preds, torch.Tensor):
            preds = preds.cpu()
        all_preds.extend(preds.tolist() if hasattr(preds, 'tolist') else list(preds))
        all_targets.extend(targets.tolist())
        correct += (torch.tensor(preds) == targets).sum().item() if not isinstance(preds, torch.Tensor) else (preds.cpu() == targets).sum().item()
        total += targets.size(0)

accuracy = correct / total
print(f"Digit addition accuracy: {accuracy:.2%}  ({correct}/{total})")

### Baselines and analysis

To understand how well your approach works, let's compare with a simple baseline:
just take the argmax prediction for each digit and add them.

Then analyze: where does your method succeed or fail compared to the baseline?

In [ ]:
# Baseline: argmax + argmax
model.eval()
correct_baseline = 0
baseline_preds = []
total = 0

with torch.no_grad():
    for img1, img2, targets in addition_loader:
        img1, img2 = img1.to(device), img2.to(device)
        pred1 = model(img1).argmax(dim=1).cpu()
        pred2 = model(img2).argmax(dim=1).cpu()
        preds = pred1 + pred2
        baseline_preds.extend(preds.tolist())
        correct_baseline += (preds == targets).sum().item()
        total += targets.size(0)

baseline_acc = correct_baseline / total
print(f"Baseline (argmax + argmax): {baseline_acc:.2%}")
print(f"Your method:               {accuracy:.2%}")
print(f"Improvement:               {accuracy - baseline_acc:+.2%}")

### Analysis

1. How much does your method improve over the argmax baseline? Why does it help (or not)?
2. For which sum values does your method work best/worst? Why might that be?
3. What is the theoretical upper bound on accuracy given your classifier's per-digit accuracy?

*Your analysis here.*

In [ ]:
# YOUR CODE HERE — analysis
# We provide a starter: per-sum-value accuracy comparison

all_preds_t = torch.tensor(all_preds)
all_targets_t = torch.tensor(all_targets)
baseline_preds_t = torch.tensor(baseline_preds)

# Per-sum accuracy
fig, ax = plt.subplots(figsize=(12, 5))
yours_per_sum = []
base_per_sum = []
sums = range(19)
for s in sums:
    mask = all_targets_t == s
    if mask.sum() > 0:
        yours_per_sum.append((all_preds_t[mask] == s).float().mean().item())
        base_per_sum.append((baseline_preds_t[mask] == s).float().mean().item())
    else:
        yours_per_sum.append(0)
        base_per_sum.append(0)

x = np.arange(19)
w = 0.35
ax.bar(x - w/2, [a*100 for a in yours_per_sum], w, label='Your method', color='steelblue')
ax.bar(x + w/2, [a*100 for a in base_per_sum], w, label='Argmax baseline', color='salmon')
ax.set_xlabel('Sum value')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Accuracy by sum value')
ax.set_xticks(x)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

# Show some example pairs with predictions
fig, axes = plt.subplots(3, 6, figsize=(14, 7))
for i in range(6):
    img1, img2, s = addition_test[i * 100]  # spread out examples
    with torch.no_grad():
        p = predict_sum(model, img1.unsqueeze(0).to(device), img2.unsqueeze(0).to(device))
    pred_val = p.item() if hasattr(p, 'item') else p[0]
    
    axes[0, i].imshow(img1.squeeze(), cmap='gray')
    axes[0, i].axis('off')
    axes[1, i].imshow(img2.squeeze(), cmap='gray')
    axes[1, i].axis('off')
    color = 'green' if pred_val == s else 'red'
    axes[2, i].text(0.5, 0.5, f'pred: {pred_val}\ntrue: {s}', 
                     ha='center', va='center', fontsize=14, color=color, fontweight='bold',
                     transform=axes[2, i].transAxes)
    axes[2, i].axis('off')

axes[0, 0].set_ylabel('Digit 1')
axes[1, 0].set_ylabel('Digit 2')
axes[2, 0].set_ylabel('Sum')
plt.suptitle('Example digit addition predictions', fontsize=13)
plt.tight_layout()
plt.show()

---
*Modern Computer Vision — Technion — Spring 2026*